# Semantic Baselines: Sentence-Transformer Cosine Similarity
Evaluates `all-MiniLM-L6-v2` and `all-mpnet-base-v2` on the Mohler dataset.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

os.environ['HF_HOME'] = r'D:\Projects 2026\genai-evaluation-and-assessment-system\.cache\huggingface'
sys.path.insert(0, '../')

from src.embeddings import cosine_scores
from src.metrics import evaluate, append_results

os.makedirs('../results/figures', exist_ok=True)

df_dev  = pd.read_csv('../data/processed/dev.csv')
df_test = pd.read_csv('../data/processed/test.csv')
print(f'DEV: {len(df_dev)} rows | TEST: {len(df_test)} rows')

## Models
- `sentence-transformers/all-MiniLM-L6-v2` — 22M params, fast
- `sentence-transformers/all-mpnet-base-v2` — 110M params, stronger

**Vocabulary note:** Unlike TF-IDF, the sentence transformer vocabulary is frozen (trained on large external corpora). No fitting on DEV/TEST is required — we only fit a thin calibration layer (LinearRegression) on the DEV cosine scores.

In [ ]:
MODELS = {
    'MiniLM-L6-v2': 'sentence-transformers/all-MiniLM-L6-v2',
    'MPNet-base-v2': 'sentence-transformers/all-mpnet-base-v2',
}

dev_cos, test_cos = {}, {}
for short, full in MODELS.items():
    print(f'Encoding {short}...')
    dev_cos[short]  = cosine_scores(full, df_dev['reference_answer'].tolist(), df_dev['student_answer'].tolist())
    test_cos[short] = cosine_scores(full, df_test['reference_answer'].tolist(), df_test['student_answer'].tolist())
    print(f'  DEV  cosine: min={dev_cos[short].min():.3f}, max={dev_cos[short].max():.3f}, mean={dev_cos[short].mean():.3f}')
    print(f'  TEST cosine: min={test_cos[short].min():.3f}, max={test_cos[short].max():.3f}, mean={test_cos[short].mean():.3f}')

## Scale to 0–5
**Method (a): Scaled** — `score = 5 × cosine` (no fitting needed)

**Method (b): Calibrated** — `LinearRegression(dev_cosines, dev_human_scores)` fitted on the DEV set only, predictions clipped to [0, 5]

In [ ]:
results_rows = []

for short, full in MODELS.items():
    print(f'\n── {short} ──')
    # (a) Scaled
    pred_s = np.clip(5.0 * test_cos[short], 0, 5)

    # (b) Calibrated — DEV only
    lr = LinearRegression().fit(dev_cos[short].reshape(-1, 1), df_dev['human_score'])
    pred_c = np.clip(lr.predict(test_cos[short].reshape(-1, 1)), 0, 5)
    print(f'  LR intercept={lr.intercept_:.3f}, coef={lr.coef_[0]:.3f}')

    # Evaluate
    m_s = evaluate(df_test['human_score'], pred_s)
    m_c = evaluate(df_test['human_score'], pred_c)
    print(f'  Scaled     — MAE={m_s["mae"]:.3f}, Pearson={m_s["pearson"]:.3f}, QWK={m_s["qwk"]:.3f}')
    print(f'  Calibrated — MAE={m_c["mae"]:.3f}, Pearson={m_c["pearson"]:.3f}, QWK={m_c["qwk"]:.3f}')

    # Save predictions
    fname = f'../results/preds_{"minilm" if "Mini" in short else "mpnet"}.csv'
    out = df_test[['row_id', 'question_id', 'human_score', 'student_answer']].copy()
    out['pred_scaled'], out['pred_calibrated'] = pred_s, pred_c
    out.to_csv(fname, index=False); print(f'  Saved {fname}')

    # Append to main_results.csv
    append_results(f'{short} (Scaled)',     m_s, '../results/main_results.csv')
    append_results(f'{short} (Calibrated)', m_c, '../results/main_results.csv')
    results_rows.append({'Method': f'{short} (Scaled)', **m_s})
    results_rows.append({'Method': f'{short} (Calibrated)', **m_c})

pd.DataFrame(results_rows).set_index('Method').round(4)

## Scatter: Human vs Predicted

In [ ]:
for short, full in MODELS.items():
    pred_s = np.clip(5.0 * test_cos[short], 0, 5)
    lr = LinearRegression().fit(dev_cos[short].reshape(-1, 1), df_dev['human_score'])
    pred_c = np.clip(lr.predict(test_cos[short].reshape(-1, 1)), 0, 5)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'{short} — TEST set predictions', fontsize=13)

    axes[0].scatter(df_test['human_score'], pred_s, alpha=0.45, label='Scaled', color='steelblue', s=30)
    axes[0].scatter(df_test['human_score'], pred_c, alpha=0.45, label='Calibrated', color='darkorange', s=30)
    axes[0].plot([0,5],[0,5],'k--',linewidth=1,label='Perfect')
    axes[0].set_xlabel('Human Score'); axes[0].set_ylabel('Predicted')
    axes[0].set_title('Human vs Predicted'); axes[0].legend(fontsize=8)

    err_s = pred_s - df_test['human_score'].values
    err_c = pred_c - df_test['human_score'].values
    axes[1].hist(err_s, bins=20, alpha=0.55, label='Scaled Error', color='steelblue', edgecolor='white')
    axes[1].hist(err_c, bins=20, alpha=0.55, label='Calibrated Error', color='darkorange', edgecolor='white')
    axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
    axes[1].set_xlabel('Error (pred − human)'); axes[1].set_title('Error Histogram'); axes[1].legend(fontsize=8)

    plt.tight_layout()
    fig_name = f'../results/figures/{"minilm" if "Mini" in short else "mpnet"}_scatter_error.png'
    plt.savefig(fig_name, dpi=150); plt.show()
    print(f'Saved {fig_name}')

## Observations

- Both sentence-transformer models achieve **Pearson ≈ 0.52** on the TEST set, roughly 3× higher than TF-IDF cosine (Pearson ≈ 0.17).
- Calibration via linear regression on the DEV set substantially reduces MAE (from ≈1.37–1.46 scaled down to ≈0.78–0.80 calibrated) by correcting the bias in cosine-to-score mapping.
- MiniLM and MPNet are nearly tied; MiniLM is preferred for inference efficiency.
- Error histograms show both models over-predict low scorers and under-predict high scorers — a known regression-to-mean effect with cosine similarity.
- These results establish the **embedding baseline ceiling** before introducing LLM-based grading in subsequent experiments.